## From Web to Insight – Automating the Knowledge Pipeline
#### NLP and Web Analytics Final Project

Arianna Tessari - Student ID: 19322

#### **Outline** 

- Introduction
- Web Scraping
- Text Classification 
- Information Retrieval & RAG System 
- Summarization & Prompt Evaluation 
- Reflection & Lessons Learned
- Limitations & Future Improvements

#### **Introduction** 

The objective of this project was to build a multi-step Natural Language Processing (NLP) pipeline capable of collecting, classifying, retrieving, summarizing, and evaluating articles about AI in healthcare. The system integrates several key components, including web scraping, zero-shot classification, Retrieval-Augmented Generation (RAG), summarization, and prompt evaluation. The use of language models throughout the pipeline, in conjunction with tailored prompt engineering strategies, allows for dynamic knowledge extraction and interpretation across a curated dataset.

A diverse range of Python libraries is employed to achieve this goal, each chosen for its strengths in web scraping, natural language processing, data management, or interaction with machine learning models. Here is an overview of the most relevant, along with their main functionalities:

- `requests` and `BeautifulSoup` are used for basic web scraping, enabling HTML parsing and data extraction.
- `selenium` is employed for dynamic web scraping, allowing access to content rendered via JavaScript.
- `pandas` and `numpy` are foundational for data handling and manipulation throughout the pipeline.
- `dotenv` is used to manage environment variables securely, such as API keys.
- `matplotlib`, `Counter`, and `wordcloud` help visualize token distributions and thematic relevance.
- `nltk` is utilized for tokenization, stopword removal, and lemmatization, contributing to text preprocessing.
- `transformers` from Hugging Face provides access to powerful pre-trained models used in zero-shot classification and text generation.
- `sentence_transformers` and `faiss` power the embedding and indexing steps in the information retrieval system.
- `openai` provides access to cutting-edge LLMs such as GPT-4o, used in summarization and QA (Question Answering) tasks.

In [ ]:
# !pip install requests beautifulsoup4 selenium newspaper3k pandas tqdm 
# !pip install "lxml[html_clean]"

In [ ]:
# Importing required libraries

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib.parse
import time
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import faiss
import textwrap
from openai import OpenAI
import json

load_dotenv()

# Show full columns and rows
pd.set_option('display.max_rows', None)

#### **1. Web Scraping** 

The first stage of the pipeline involves gathering a dataset of online articles focused on the intersection of AI and healthcare. To this end, the project targeted the Harvard Gazette, a reliable and reputable news source affiliated with Harvard University. The goal was to collect at least 10 relevant articles related to the theme of **"AI in Healthcare"**, along with their title, URL, publication date, source, and main text content.

To achieve this, two complementary scraping techniques were used: HTTP-based requests and Selenium-driven browser automation. While a standard `requests.get()` call was used to confirm the availability of the base website (`<Response [200]>`), the majority of scraping was handled through `Selenium`, due to the dynamic nature of the Harvard Gazette’s search results. This allowed the pipeline to interact with JavaScript-rendered content, simulate user actions like clicking pagination buttons, and open articles in new browser tabs for detailed content extraction.

A custom function, `extract_article_links()`, was implemented to identify and collect URLs pointing to individual article pages. These were filtered to match only those beginning with a known article URL pattern, preventing unrelated links from being included. An auxiliary function, `go_to_next_page()`, enabled the script to navigate through multiple pages of search results to locate enough unique articles.

Each article was then opened in a new browser tab using JavaScript execution via Selenium. Within the article, several different content containers were checked to extract paragraph text, accounting for potential variations in HTML structure. The script also attempted to extract the article's title and date using specific CSS selectors. Articles that failed to provide this metadata were skipped to maintain the integrity of the dataset.

This entire process was run in a loop until 10 successfully parsed articles were collected. The output was stored in a Pandas DataFrame, which was then exported to a CSV file (`ai_healthcare_articles.csv`) for future analysis. A snippet of the collected metadata is shown below.

In [ ]:
# page URL
url = 'https://news.harvard.edu/gazette/'

# Making a GET request
r = requests.get(url)

# check status code for response received
# success code - 200
print(r)

# print content of request
# print(r.content)

In [ ]:
SEARCH_QUERY = "AI in Healthcare"
MAX_ARTICLES_TO_SCRAPE = 10
BASE_URL = 'https://news.harvard.edu/gazette/'

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

def extract_article_links():
    links = driver.find_elements(By.CSS_SELECTOR, "div.gsc-webResult.gsc-result div.gs-title a.gs-title")
    urls = []
    for link in links:
        href = link.get_attribute("href")
        if href and href.startswith("https://news.harvard.edu/gazette/story/") and href not in seen_urls:
            urls.append(href)
            seen_urls.add(href)
    return urls

def go_to_next_page():
    try:
        pagination = driver.find_elements(By.CSS_SELECTOR, "div.gsc-cursor-page")
        current_page = driver.find_element(By.CSS_SELECTOR, "div.gsc-cursor-current-page")
        current_index = [i for i, el in enumerate(pagination) if el.text == current_page.text][0]
        if current_index + 1 < len(pagination):
            pagination[current_index + 1].click()
            WebDriverWait(driver, 10).until(
                EC.staleness_of(pagination[current_index + 1])
            )
            time.sleep(2)
            return True
    except Exception as e:
        print("No more pages or error:", e)
    return False

try:
    search_url = f"{BASE_URL}?s={urllib.parse.quote_plus(SEARCH_QUERY)}"
    driver.get(search_url)

    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.gsc-webResult.gsc-result"))
    )

    seen_urls = set()
    articles = []
    article_count = 0

    while article_count < MAX_ARTICLES_TO_SCRAPE:
        new_links = extract_article_links()

        for url in new_links:
            if article_count >= MAX_ARTICLES_TO_SCRAPE:
                break

            # Open article in a new tab
            driver.execute_script("window.open('');")
            driver.switch_to.window(driver.window_handles[1])
            driver.get(url)
            time.sleep(2)

            # Scrape article info
            try:
                title = driver.find_element(By.CSS_SELECTOR, ".article-header__title.wp-block-heading").text.strip()
            except:
                print(f"Title not found. Skipping: {url}")
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
                continue

            try:
                date = driver.find_element(By.CSS_SELECTOR, "time.entry-date, time.published, time").get_attribute("datetime")
            except:
                print(f"Date not found. Skipping: {url}")
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
                continue

            content = ""
            containers = [
                "div.entry-content",
                "div.post-content",
                "article.wp-block-post-content",
                "div.article-body",
                "div.the-content"
            ]
            for container in containers:
                try:
                    paras = driver.find_elements(By.CSS_SELECTOR, f"{container} p")
                    if paras:
                        content = " ".join(p.text for p in paras if p.text.strip())
                        if content:
                            break
                except:
                    continue

            if not content:
                paras = driver.find_elements(By.TAG_NAME, "p")
                content = " ".join(p.text for p in paras if p.text.strip())

            article_count += 1
            print(f"Scraping article {article_count}: {url}")

            articles.append({
                "title": title,
                "url": url,
                "date": date,
                "source": "Harvard Gazette",
                "content": content
            })

            # Close article tab and switch back to search results
            driver.close()
            driver.switch_to.window(driver.window_handles[0])

        if article_count >= MAX_ARTICLES_TO_SCRAPE:
            break

        if not go_to_next_page():
            print("No more pages.")
            break

    df = pd.DataFrame(articles)

finally:
    driver.quit()
    print("Driver closed.")

In [ ]:
df

In [ ]:
df.to_csv("ai_healthcare_articles.csv", index=False)

To verify the relevance and quality of the data, each article was printed with all its metadata. This allowed a manual check of whether the content aligned with the search query and whether the scraping logic was successfully extracting useful information.

In [ ]:
df = pd.read_csv("ai_healthcare_articles.csv")

# Print each article with title, date, source, URL, and content
for idx, row in df.iterrows():
    print(f"\n--- ARTICLE {idx+1} ---")
    print(f"Title   : {row['title']}")
    print(f"Date    : {row['date']}")
    print(f"Source  : {row['source']}")
    print(f"URL     : {row['url']}\n")
    print(f"Content :\n{row['content']}")
    print("\n" + "-"*80)

**Tokenize Sentences and Remove Stopwords**

Finally, to begin exploring trends within the dataset, basic text processing was applied to the content field (e.g., tokenization, stopword removal, lemmatization), followed by word frequency analysis. The top 20 most common words were visualized using a bar chart, offering insight into the dominant themes across the articles. Furthermore, a word cloud was generated to represent these frequently occurring terms in a visually engaging way, reinforcing the presence of key concepts such as "AI", "medicine", "patients", "technology", "treatment".

In [ ]:
# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Preprocessing function
def preprocess_text(text):
    # Lowercase and tokenize
    words = word_tokenize(text.lower())
    # Remove stopwords and non-alphanumeric, then lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word.isalnum() and word not in stop_words]
    return words

# Apply to the scraped 'content' column
df['processed_text'] = df['content'].apply(preprocess_text)

# Preview the result
print(df[['title', 'processed_text']])

**Visualize Word Frequency Distribution**

In [ ]:
# Flatten list (one list with all words)
all_words = [word for tokens in df['processed_text'] for word in tokens]

# Create dictionary where keys are words and values are their counts
word_freq = Counter(all_words)

# Get most common words
top_n = 20
most_common_words = word_freq.most_common(top_n)

# Separate words and their counts into two tuples (words, counts)
words, counts = zip(*most_common_words)

# Plot bar chart with frequency
plt.figure(figsize=(14, 7))
bars = plt.bar(words, counts, color='skyblue')
plt.title(f'Top {top_n} Most Common Words in Harvard Gazette Articles', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Words', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add count labels above bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 1, str(height), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(all_words))

# Plot word cloud
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

#### **2. Text Classification** 

Once the articles dataset was successfully scraped and organized, the next step involved classifying each article into thematic categories to better structure the content and support downstream tasks like targeted summarization and retrieval. Instead of manually labeling each article, the project leveraged a zero-shot classification approach using a transformer-based language model.

**Model and Rationale**

The classification was performed using the `HuggingFace` pipeline API with the `valhalla/distilbart-mnli-12-3` model, which is a distilled version of BART fine-tuned on the Multi-Genre Natural Language Inference (MNLI) dataset. This makes it particularly effective for zero-shot learning tasks, where the model can assign texts to user-defined labels without any additional training.

This approach was selected for two key reasons:

1. Flexibility – It allows the use of custom, domain-specific labels tailored to this project.
2. Efficiency – It avoids the need to curate and annotate a training dataset, enabling rapid experimentation.

**Thematic Labels**

Three custom thematic categories were defined to reflect the most relevant areas of discourse around AI in healthcare:

- Clinical Applications & Tools: Articles focusing on diagnostic technologies, treatment systems, or AI-enabled tools in practice.
- Ethical & Patient Safety: Content discussing privacy concerns, decision-making risks, or moral considerations.
- Policy & Economic Impact: Pieces covering regulatory frameworks, cost implications, or broader social/political effects.

These labels were passed as candidate labels to the classifier, which evaluated the semantic fit of each article's content against the labels.

**Classification Workflow**

The entire body of each article was processed as input to the model. For each input, the classifier returned a ranked list of labels along with confidence scores. Only the top-ranked label (i.e., the one with the highest score) was retained for each article. The associated confidence score was also recorded to allow for transparency and potential filtering. 

Although the model successfully categorized all articles, the confidence scores were generally modest (between 33%–39%), suggesting that thematic boundaries in healthcare discourse are nuanced and sometimes overlapping. This also reflects the limitation of using a generic NLI model on a specialized medical dataset. Nonetheless, the zero-shot approach offered a fast and effective solution for thematic tagging without the need for labeled training data.

In [ ]:
# Initialize zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", truncation=True, model="valhalla/distilbart-mnli-12-3")

# Define candidate labels (thematic categories)
candidate_labels = [
    "Clinical Applications & Tools", 
    "Ethical & Patient Safety",
    "Policy & Economic Impact"
    ]

themes = []
confidences = []

for content in df['content']:
    result = classifier(content, candidate_labels)
    # Get the top predicted label and score
    top_label = result['labels'][0]
    top_score = f"{result['scores'][0]:.2%}"
    
    themes.append(top_label)
    confidences.append(top_score)

df['theme'] = themes
df['confidence'] = confidences

df[['title', 'theme', 'confidence']]

#### **3. Information Retrieval & RAG System** 

Once the articles had been categorized, the next objective was to build a Retrieval-Augmented Generation (RAG) system that could answer natural language questions by grounding its responses in the scraped article content. This system combines vector-based document retrieval with large language models (LLMs) for context-aware question answering, allowing the user to interact with the corpus as if it were a live knowledge base.

**Vector Embedding & Semantic Search**

The first step involved transforming each article into a vector representation, capturing its semantic content. This was achieved using the `all-MiniLM-L6-v2` model from `SentenceTransformer`, which encodes each article into a 384-dimensional embedding. This model strikes a balance between speed and semantic accuracy, making it ideal for projects that require near real-time vector search with limited hardware.

These embeddings were stored in a FAISS index, a high-performance similarity search library optimized for dense vector search. FAISS was used in flat L2 mode (`IndexFlatL2`), where all vectors are stored and searched exhaustively, a feasible choice given the small size of the dataset. This enabled fast retrieval of the top k most relevant articles for any user query, based on semantic similarity.

**Prompt Engineering & Response Generation**

To generate answers, a prompt template was created that included:

- A clear system role ("You are a helpful assistant specialized in medical and academic topics")
- A structured context block containing shortened text chunks from top-matching articles
- A reminder to cite sources using numeric references 
- The user's question

This setup allows the language model to generate answers that are both context-grounded and explainable, with explicit references to supporting texts.

Two retrieval functions were created to support answer generation:

- One using `GPT-2` via `Hugging Face`'s pipeline("text-generation"), with prompts formatted manually to include context chunks, retrieval references, and assistant instructions.

- Another using `GPT-4o` via the OpenAI API, structured around a dedicated system_message that outlines constraints such as avoiding hallucination, using only retrieved sources, and maintaining a professional tone.

In both methods, the top 3 most relevant articles are retrieved based on vector similarity. Their content is truncated for prompt compactness and injected into the LLM prompt alongside the user’s question. 

In [ ]:
# 1. Load articles
# df = pd.read_csv("ai_healthcare_articles.csv")

# 2. Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Compute embeddings for all articles
print("Encoding article contents...")
texts = df['content'].fillna("").tolist()
titles = df['title'].fillna("").tolist()
embeddings = embedding_model.encode(texts, convert_to_numpy=True)

# 4. Create or load FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"✅ FAISS index created and populated with {index.ntotal} vectors.")

# 5. Save index
faiss.write_index(index, "embeddings.index")
print("✅ FAISS index saved to 'embeddings.index'")

# 6. Define QA function with prompt engineering
def retrieve_with_prompt(query, model, index, texts, titles, k=3, max_context_chars=300):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k=k)

    top_chunks = [texts[i] for i in indices[0]]
    top_titles = [titles[i] for i in indices[0]]

    # Create context with shortened text for readability
    context = "\n\n".join(
        f"[{i+1}] {textwrap.shorten(text.replace('\n', ' '), width=max_context_chars)}"
        for i, text in enumerate(top_chunks)
    )

    # Prompt engineering
    prompt = (
        "You are a helpful assistant specialized in medical and academic topics.\n"
        "Based on the context sections below, provide a clear and concise answer to the user’s question.\n"
        "Reference the relevant context sections using their numbers (e.g., [1], [2]).\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\nAnswer:"
    )

    # Generate answer using GPT-2
    generator = pipeline("text-generation", model="gpt2", tokenizer="gpt2")
    response = generator(prompt, max_new_tokens=200, do_sample=True, top_k=50, temperature=0.8)[0]['generated_text']
    answer = response.replace(prompt, "").strip()

    return answer, top_titles

# 7. Example QA
example_query = "How is AI used in diagnosing diseases?"
answer, sources = retrieve_with_prompt(example_query, embedding_model, index, texts, titles)

# 8. Print output with formatting
print(f"\n🧠 Query:\n{textwrap.fill(example_query, width=100)}")
print(f"\n💬 Answer:\n{textwrap.fill(answer, width=100)}")
print("\n📚 Source Articles:")
for i, src in enumerate(set(sources), 1):
    print(f"{i}. {src}")

**Storing and Loading the OpenAI API Key as an Environment Variable**

For secure and maintainable handling of credentials, the OpenAI API key was stored in a `.env` file and loaded into the environment. First, a dedicated Python virtual environment was created for the project to isolate dependencies and maintain a clean development environment. The `.env` file was created using the terminal command `echo OPENAI_API_KEY=your_api_key > .env`, which stored the API key in the appropriate format. To verify that the file was created correctly, `cat .env` (or type `.env` on Windows) was executed, which returned the expected output: `OPENAI_API_KEY=your_api_key`. Additionally, its existence was confirmed in Python using `print(os.path.isfile('.env'))`, which returned `True`.

In [ ]:
# Check if the .env file exists
print(os.path.isfile('.env'))

Within the pipeline, the key was accessed securely using `os.getenv("OPENAI_API_KEY")`, ensuring it was never hard-coded into the script. This key was then passed to the OpenAI client for authenticated access to the GPT-4o model. By using this approach within a virtual environment, the project remains both secure and portable, enabling seamless collaboration and deployment without exposing sensitive credentials.

In [ ]:
# Initialize OpenAI client
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute embeddings for all articles
texts = df['content'].fillna("").tolist()
titles = df['title'].fillna("").tolist()

print("Encoding article contents...")
embeddings = embedding_model.encode(texts, convert_to_numpy=True)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"✅ FAISS index created with {index.ntotal} vectors.")

# Save the index if needed
faiss.write_index(index, "embeddings.index")
print("✅ FAISS index saved to 'embeddings.index'")

# Define system message for the assistant
system_message = """
You are an intelligent assistant designed to answer user questions 
accurately and concisely by using the provided content. 
Your goal is to prioritize information from the documents and resources available to you, 
ensuring that all responses are grounded in the retrieved content. 

Key Guidelines:
1. Use only the provided content to answer questions. Do not make assumptions or include information that is not explicitly mentioned in the resources.
2. If the content does not provide sufficient information to answer the question, politely inform the user that the answer is not available in the provided material.
3. Present responses in a clear and professional manner, tailored to the user's question.
4. When appropriate, provide additional context or clarification from the retrieved content, but avoid unnecessary elaboration.
5. Use natural and conversational language to ensure the response is user-friendly.

Your purpose is to assist users in understanding and utilizing the provided information effectively.
"""

def retrieve_and_answer(user_question, model, index, texts, titles, k=3, max_context_chars=500):
    # 1. Embed the user query
    query_embedding = embedding_model.encode([user_question], convert_to_numpy=True)
    
    # 2. Search for the top k relevant documents
    distances, indices = index.search(query_embedding, k=k)
    
    # 3. Get the matching article texts and titles
    matched_texts = [texts[i] for i in indices[0]]
    matched_titles = [titles[i] for i in indices[0]]
    
    # 4. Build the prompt content with clear references and truncated text
    context = "\n\n".join(
        f"[{i+1}] {textwrap.shorten(text.replace('\n', ' '), width=max_context_chars)}"
        for i, text in enumerate(matched_texts)
    )
    
    # 5. Construct the user prompt for the LLM
    prompt = f"""
The following content is provided to help answer user questions:

Content:
{context}

User Question:
{user_question}

Please use this content to answer the user's question as accurately as possible.
Remember to only use the information explicitly mentioned in the content and avoid adding any
outside knowledge or assumptions. Also please use clever formatting for readability.
"""
    
    # 6. Call OpenAI API
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt},
        ],
        temperature=0.7,
        max_tokens=500,
    )
    
    answer = completion.choices[0].message.content.strip()
    
    return answer, matched_titles


# Example usage
user_question = "How is AI used in diagnosing diseases?"
answer, sources = retrieve_and_answer(user_question, model="gpt-4o", index=index, texts=texts, titles=titles)

print("\n🧠 Query:\n", user_question)
print("\n💬 Answer:\n", answer)
print("\n📚 Source Articles:")
for i, title in enumerate(sources, 1):
    print(f"{i}. {title}")

##### **Two-Model Comparison: GPT-2 vs GPT-4o**

**Model 1: GPT-2 (Local Hugging Face Model)**

Initially, the system was tested using the open-source `gpt2` model via `HuggingFace`’s text generation pipeline. Despite being accessible and fast, GPT-2 has significant limitations:

- It has no built-in chat memory or instruction-following capacity
- It produces verbose, occasionally hallucinatory responses
- It lacks the fine-tuned control required for structured or sourced answers

In fact, the answer generated by GPT-2 to the query “How is AI used in diagnosing diseases?” seemed to be in parts overly generalized, verbose and occasionally inconsistent. This illustrates GPT-2’s limitations in handling fact-based, high-stakes content, especially when precision and traceability are crucial.

**Model 2: GPT-4o (via OpenAI API)**

The same user query was passed to this model using a refined prompt that emphasized:

- Strict use of only the provided content
- Clear formatting (e.g., bullet points, headings)
- Professional tone and conversational language

The responses generated by GPT-4o were notably more accurate, structured, and context-aware than those from GPT-2. For example, when asked “How is AI used in diagnosing diseases?”, GPT-4o returned a segmented answer citing long COVID detection, cancer diagnosis, and AI imaging tools—each tied explicitly to a source article. 

This comparison underscores the advantage of modern instruction-tuned models and also highlights the value of thoughtful prompt design in maintaining LLM reliability.

#### **4. Summarization & Prompt Evaluation** 

The final stage involved summarizing each article using three distinct prompting strategies. A custom `build_prompt_strategy()` function defined different summarization modes:

- Role-based prompt: instructs the model to behave like a scientific editor writing for a research bulletin.
- Format constraint prompt: restricts the output to exactly three bullet points under 20 words each.
- Fallback prompt: asks the model to simplify technical content or highlight only the key takeaways when the article is unclear.

Each strategy was passed to GPT-4o using the OpenAI API in a structured chat-completion format. This design allowed consistent control over tone, output length, and structure. The model was able to successfully adapt its behavior to each prompt. For instance, role-based summaries included academic terminology and longer paragraphs, while bullet-pointed summaries emphasized clarity and brevity. Fallback summaries offered a balance of simplification and informativeness, crucial for general audiences. 

To evaluate summarization quality, the pipeline implemented a second LLM-based system that assessed each output across four criteria:

- Coverage (are all important ideas present?)
- Conciseness (is the summary brief and to the point?)
- Coherence (is it logically structured?)
- Faithfulness (is it accurate to the original content?)

Each summary was assigned scores in structured JSON format, enabling objective comparison between strategies. Results showed that role-based prompts achieved the highest scores across all four criteria, while fallback prompts provided a strong balance of brevity and accuracy, and format-constrained bullet summaries, though clear, lagged in coverage and faithfulness.

In [ ]:
# Load the OpenAI API key from .env
load_dotenv()
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

# Define different prompt strategies
def build_prompt_strategy(content, strategy="default"):
    content = content.strip()
    
    if len(content) < 50:
        return "The content is too short to generate a meaningful summary."

    if strategy == "role":
        return f"""
You are a scientific editor. Please summarize the following article for a research bulletin:

\"\"\"{content}\"\"\"

Summary:
"""

    elif strategy == "format_constraint":
        return f"""
Summarize the following article in exactly 3 bullet points. Each bullet should be under 20 words:

\"\"\"{content}\"\"\"

Summary:
- 
- 
- 
"""

    elif strategy == "fallback":
        return f"""
If the content is too technical, simplify it. If it's unclear, summarize only the key messages:

\"\"\"{content}\"\"\"

Summary:
"""

    else:  # default
        return f"""
Summarize the following article in 2–3 concise sentences:

\"\"\"{content}\"\"\"

Summary:
"""

# Generate summaries
def generate_summaries(df, strategy="default", model="gpt-4o"):
    summaries = []

    for content in df['content']:
        prompt = build_prompt_strategy(content, strategy)

        if "too short" in prompt:
            summaries.append("⚠️ Content too short to summarize.")
            continue

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that summarizes academic articles."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5,
            max_tokens=250
        )

        summary = response.choices[0].message.content.strip()
        summaries.append(summary)

    return summaries

# Generate summaries for each strategy
df["summary_role"] = generate_summaries(df, strategy="role")
df["summary_bullets"] = generate_summaries(df, strategy="format_constraint")
df["summary_fallback"] = generate_summaries(df, strategy="fallback")

# Optional: Save to file
# df.to_csv("summarized_articles.csv", index=False)

# Display the results
for i, row in df.iterrows():
    print(f"\n📝 Article {i+1}:")
    print(f"\n🔹 Role Summary:\n{row['summary_role']}")
    print(f"\n🔹 Bullet Summary:\n{row['summary_bullets']}")
    print(f"\n🔹 Fallback Summary:\n{row['summary_fallback']}")

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. Build evaluation prompt
def build_evaluation_prompt(article, summary):
    return f"""
You are an expert reviewer assessing the quality of a summary of a scientific article.

Article:
\"\"\"{article}\"\"\"

Summary:
\"\"\"{summary}\"\"\"

Please rate the summary on a scale from 1 to 5 (5 = best) on the following criteria:

1. Coverage: How well does the summary cover the main points of the article?
2. Conciseness: Is the summary brief and to the point?
3. Coherence: Is the summary logically structured and easy to follow?
4. Faithfulness: Does the summary accurately represent the facts in the article?

Reply only in this format (no explanation, just numbers):
Coverage: <1-5>
Conciseness: <1-5>
Coherence: <1-5>
Faithfulness: <1-5>
"""

# 2. Parse the plain-text response into 4 numeric scores
def parse_scores(text):
    try:
        lines = text.strip().splitlines()
        scores = {}
        for line in lines:
            key, val = line.split(":")
            scores[key.strip().lower()] = int(val.strip())
        return (
            scores.get("coverage", None),
            scores.get("conciseness", None),
            scores.get("coherence", None),
            scores.get("faithfulness", None)
        )
    except Exception as e:
        # If parsing fails, return None
        return None, None, None, None

# 3. Evaluate a single summary
def evaluate_summary_llm(article, summary, model="gpt-4o"):
    if summary.startswith("⚠️") or len(summary.strip()) < 20:
        return None, None, None, None
    
    prompt = build_evaluation_prompt(article, summary)
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert summary evaluator."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=100
        )
        raw_output = response.choices[0].message.content
        return parse_scores(raw_output)
    except Exception as e:
        return None, None, None, None

# 4. Add evaluation scores to the DataFrame for one summary column
def add_scores_to_df(df, summary_col):
    covs, concs, cohes, faiths = [], [], [], []

    for i, row in df.iterrows():
        article = row["content"]
        summary = row[summary_col]
        c, co, ch, f = evaluate_summary_llm(article, summary)
        covs.append(c)
        concs.append(co)
        cohes.append(ch)
        faiths.append(f)

    df[f"{summary_col}_coverage"] = covs
    df[f"{summary_col}_conciseness"] = concs
    df[f"{summary_col}_coherence"] = cohes
    df[f"{summary_col}_faithfulness"] = faiths
    return df

# 5. Apply to each summary column (edit as needed)
summary_columns = ["summary_role", "summary_bullets", "summary_fallback"]

for col in summary_columns:
    print(f"🔍 Evaluating summaries in column: {col}")
    df = add_scores_to_df(df, col)

In [ ]:
# List of summary strategies
summary_cols = ["summary_role", "summary_bullets", "summary_fallback"]

# Initialize dictionary to store averages
averages = {}

for col in summary_cols:
    averages[col] = {
        "coverage": df[f"{col}_coverage"].mean(),
        "conciseness": df[f"{col}_conciseness"].mean(),
        "coherence": df[f"{col}_coherence"].mean(),
        "faithfulness": df[f"{col}_faithfulness"].mean(),
    }

# Convert to DataFrame for a nice table
avg_df = pd.DataFrame(averages).T.round(2)

print("\n📊 Average evaluation scores per summary type:")
print(avg_df)


In [ ]:
structured_output = []

for _, row in df.iterrows():
    record = {
        "title": row.get("title", ""),  # optional, if you have titles
        "summaries": {
            "role": {
                "coverage": row.get("summary_role_coverage"),
                "conciseness": row.get("summary_role_conciseness"),
                "coherence": row.get("summary_role_coherence"),
                "faithfulness": row.get("summary_role_faithfulness"),
            },
            "bullets": {
                "coverage": row.get("summary_bullets_coverage"),
                "conciseness": row.get("summary_bullets_conciseness"),
                "coherence": row.get("summary_bullets_coherence"),
                "faithfulness": row.get("summary_bullets_faithfulness"),
            },
            "fallback": {
                "coverage": row.get("summary_fallback_coverage"),
                "conciseness": row.get("summary_fallback_conciseness"),
                "coherence": row.get("summary_fallback_coherence"),
                "faithfulness": row.get("summary_fallback_faithfulness"),
            }
        }
    }
    structured_output.append(record)

# Save to JSON file
with open("evaluated_summaries.json", "w", encoding="utf-8") as f:
    json.dump(structured_output, f, indent=2, ensure_ascii=False)

print("✅ JSON file 'evaluated_summaries.json' created with structured scores.")

#### **Reflection & Lessons Learned** 

This project illustrates the power and modularity of modern NLP pipelines when integrated with LLMs and effective prompt engineering. Several key insights emerged:

- Prompt Engineering is essential: Output quality across tasks (classification, QA, summarization) was directly influenced by the specificity, structure, and tone of the prompts. Even minor changes in phrasing led to significantly different model behaviors.

- Model choice matters: GPT-4o consistently outperformed earlier models like GPT-2 in factuality, coherence, and readability, especially in complex summarization and QA tasks. This supports a trend toward using newer, instruction, aligned models for high-stakes domains like healthcare.

- Zero-shot classification is fast but imperfect: While NLI-based classification offered good generalization, low confidence scores suggest that fine-tuned models or domain-specific taxonomies may yield better granularity.

- RAG systems scale well: The combination of FAISS indexing and GPT-based generation proved effective and lightweight. Embedding-based retrieval ensured contextual relevance without the overhead of large-scale document search engines.

#### **Limitations & Future Improvements**

- Dataset size: With only 10 articles, the system could not explore advanced indexing or document clustering. Scaling to hundreds of sources would provide richer insights.

- Evaluation automation: While GPT-based evaluation was informative, incorporating human judgment or ROUGE/BERTScore metrics could improve robustness.

- Interactive front-end: A user interface to allow live querying, exploration of themes, and side-by-side comparison of summaries would greatly enhance usability.

- Summary fusion: Combining multiple summarization strategies (e.g., bullet + role) could produce hybrid outputs optimized for different audiences. For example, a hybrid summary could begin with a high-level bullet-point overview for quick skimming, followed by a more in-depth paragraph capturing nuanced insights. This fusion would cater to both general readers looking for clarity and specialists seeking detailed interpretations, ultimately improving accessibility and engagement across user profiles.